🧩 Different Ways of Structuring Classes in Composition
Composition gives you flexibility in how you wire classes together. In enterprise coding, there are several structuring styles you can use depending on the problem:

1. Constructor Injection (Most Common)
Pass dependencies via the constructor.

Ensures the object is always created with its required components.

In [1]:
class Engine:
    def start(self): return "Engine started"

class Car:
    def __init__(self, engine):
        self.engine = engine   # injected via constructor
    def drive(self):
        print(self.engine.start())
        print("Car is driving")

car = Car(Engine())   # composition via constructor


✅ Use case: ETL pipelines where Pipeline must always have Ingestion, Transformation, and Storage objects.

2. Setter Injection
Provide dependencies after object creation using setter methods.

More flexible, but risks having incomplete objects if setters aren’t called.

In [2]:
class Car:
    def set_engine(self, engine):
        self.engine = engine
    def drive(self):
        print(self.engine.start())
        print("Car is driving")

car = Car()
car.set_engine(Engine())   # composition via setter


✅ Use case: Configurable enterprise services where components may be swapped dynamically (e.g., logging, monitoring).

3. Interface-Based Composition
Define abstract interfaces and inject implementations.

Promotes loose coupling and testability.

In [6]:
from abc import ABC, abstractmethod

class Engine(ABC):
    @abstractmethod
    def start(self): pass

class ElectricEngine(Engine):
    def start(self): return "Electric engine started"

class DieselEngine(Engine):
    def start(self): return "Diesel engine roaring"

class Car:
    def __init__(self, engine: Engine):
        self.engine = engine
    def drive(self):
        print(self.engine.start())
        print("Car is driving")

car = Car(ElectricEngine())   # can swap engine easily


✅ Use case: Enterprise ETL where you may switch between Kafka ingestion vs Event Hub ingestion without changing pipeline code.

4. Aggregation (Collection of Components)
A class holds multiple components in a list/dict and orchestrates them.

Useful when you need to run multiple strategies or modules.

In [4]:
class Pipeline:
    def __init__(self, steps):
        self.steps = steps
    def run(self, data):
        for step in self.steps:
            data = step.run(data)
        return data


✅ Use case: Data pipelines with multiple transformation stages (cleaning, enrichment, aggregation).

5. Decorator Composition
Wrap one component with another to extend behavior.

Often used in logging, monitoring, or middleware.

In [8]:
from abc import ABC, abstractmethod

class Engine(ABC):
    @abstractmethod
    def start(self): pass

class PetrolEngine(Engine):
    def start(self):
        return "Petrol engine started"

class LoggingEngine:
    def __init__(self, engine: Engine):
        self.engine = engine
    def start(self):
        print("Logging: Engine start called")
        return self.engine.start()

class Car:
    def __init__(self, engine: Engine):
        self.engine = engine
    def drive(self):
        print(self.engine.start())
        print("Car is driving")

# ✅ Use a concrete subclass
car = Car(LoggingEngine(PetrolEngine()))
car.drive()



Logging: Engine start called
Petrol engine started
Car is driving


✅ Use case: Enterprise services where you add audit logging or metrics collection without modifying core logic.

🔹 Summary
Constructor Injection → Mandatory dependencies (ETL pipeline stages).

Setter Injection → Optional/configurable dependencies (logging, monitoring).

Interface-Based → Swap implementations (different ingestion/transformation engines).

Aggregation → Multiple components orchestrated (multi-step workflows).

Decorator → Extend behavior (logging, caching, retry logic).